# Clean — PhysioNet Sepsis 2019

**Vấn đề từ EDA**: dữ liệu time-series theo giờ, nhiều NaN ở xét nghiệm không đo mỗi giờ — khó ghép với các bộ bảng tĩnh khác.

**Quyết định**: Aggregate về **1 dòng/bệnh nhân** — lấy giá trị xét nghiệm/sinh hiệu **gần nhất (không rỗng)** trong toàn bộ thời gian nằm viện, Age/Gender cố định, nhãn `ever_sepsis` = max(SepsisLabel) qua các giờ.

**Output**: `Clean_Data/tabular/physionet_sepsis_clean.csv` (~40.336 dòng, 1 dòng/bệnh nhân)

Xử lý toàn bộ 40.336 file — có thể mất vài phút.

In [1]:
import pandas as pd
from pathlib import Path
import time

ROOT = Path(r'D:\AI_08_V1\Data\tabular\physionet_sepsis_2019')
OUT = Path(r'D:\AI_08_V1\Clean_Data\tabular')

# Cột lấy giá trị gần nhất (không rỗng); Age/Gender lấy giá trị đầu (cố định suốt lượt nằm viện)
last_val_cols = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
                  'Hct', 'Hgb', 'WBC', 'Platelets', 'PTT', 'Fibrinogen',
                  'Bilirubin_total', 'Bilirubin_direct', 'AST', 'Alkalinephos', 'Creatinine']

def process_file(f):
    d = pd.read_csv(f, sep='|')
    row = {'patient_id': f.stem, 'source_set': f.parent.name,
           'Age': d['Age'].iloc[0], 'Gender': d['Gender'].iloc[0],
           'n_hours': len(d), 'ever_sepsis': int(d['SepsisLabel'].max())}
    for col in last_val_cols:
        s = d[col].dropna()
        row[col] = s.iloc[-1] if len(s) else None
    return row

files = list((ROOT / 'training_setA').glob('*.psv')) + list((ROOT / 'training_setB').glob('*.psv'))
print('Tổng số file cần xử lý:', len(files))

t0 = time.time()
rows = [process_file(f) for f in files]
print(f'Thời gian xử lý: {time.time()-t0:.1f}s')

df_clean = pd.DataFrame(rows)
print('Shape kết quả:', df_clean.shape)
df_clean.head()

Tổng số file cần xử lý: 40336


Thời gian xử lý: 358.6s


Shape kết quả: (40336, 24)


,patient_id,source_set,Age,Gender,n_hours,ever_sepsis,HR,O2Sat,Temp,SBP,...,Hgb,WBC,Platelets,PTT,Fibrinogen,Bilirubin_total,Bilirubin_direct,AST,Alkalinephos,Creatinine
0,p000001,training_setA,83.14,0,54,0,84.0,85.0,36.33,78.0,...,12.2,14.7,338.0,NaN,NaN,0.3,NaN,16.0,98.0,0.7
1,p000002,training_setA,75.91,0,23,0,55.0,95.0,36.11,114.0,...,9.7,11.0,158.0,NaN,NaN,NaN,NaN,NaN,NaN,2.5
2,p000003,training_setA,45.82,0,48,0,78.0,97.0,37.11,138.0,...,11.0,8.7,486.0,29.5,NaN,NaN,NaN,NaN,NaN,0.8
3,p000004,training_setA,65.71,0,29,0,108.0,98.0,36.72,115.0,...,8.3,7.6,144.0,22.3,NaN,NaN,NaN,NaN,NaN,0.8
4,p000005,training_setA,28.09,1,48,0,80.0,97.0,36.33,134.0,...,15.5,4.7,288.0,29.0,NaN,0.6,NaN,30.0,80.0,0.6


In [2]:
# Lưu file
out_path = OUT / 'physionet_sepsis_clean.csv'
df_clean.to_csv(out_path, index=False)
print(f'Đã lưu: {out_path} ({len(df_clean)} dòng, {df_clean.shape[1]} cột)')
print('\nPhân bố ever_sepsis:')
print(df_clean['ever_sepsis'].value_counts(normalize=True).round(3))
print('\nTỷ lệ missing theo cột xét nghiệm (sau khi lấy giá trị gần nhất):')
print((df_clean[last_val_cols].isna().mean() * 100).round(1).sort_values(ascending=False))

Đã lưu: D:\AI_08_V1\Clean_Data\tabular\physionet_sepsis_clean.csv (40336 dòng, 24 cột)

Phân bố ever_sepsis:
ever_sepsis
0    0.927
1    0.073
Name: proportion, dtype: float64

Tỷ lệ missing theo cột xét nghiệm (sau khi lấy giá trị gần nhất):
Bilirubin_direct    94.9
Fibrinogen          88.8
Alkalinephos        64.9
Bilirubin_total     64.7
AST                 64.4
PTT                 49.8
DBP                 18.4
WBC                  6.5
Platelets            6.4
Hgb                  6.1
Hct                  5.7
Creatinine           5.1
SBP                  0.7
Temp                 0.7
MAP                  0.3
Resp                 0.2
O2Sat                0.0
HR                   0.0
dtype: float64
